<a href="https://colab.research.google.com/github/PIER-202621036/proyecto-gestion-vacaciones/blob/main/notebooks%20/02_vacaciones_indemnizadas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from IPython.display import display
from datetime import date

# 1. Configurar Pandas para que NO parta la tabla ni oculte columnas en la consola
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


Cargar Data Excel

In [2]:
URL_EXCEL = (
    "https://raw.githubusercontent.com/"
    "PIER-202621036/proyecto-gestion-vacaciones/"
    "main/data/empleados_vacaciones.xlsx"
)

# Hojas que usa Python (las demas del Excel son material de gestion)
EXCEL_HOJAS_PRINCIPALES = [
    "Empleados",
    "Puestos",
    "Vacaciones",
    "Programacion_Vacaciones",
    "Ofertas_Compensacion",
    "Parametros_Vacaciones",
    "Parametros_Criticidad",
    "Niveles_Criticidad"
]

# Unicas columnas que pueden estar vacias (segun el Diccionario)
NULOS_PERMITIDOS = ["motivo", "fecha_estado"]

negocio_excel = {}
negocio_parametros = {}

# Cargar todas las hojas del Excel
"""Lee el Excel y devuelve los datos ORIGINALES, sin limpiar.
    El resultado es un diccionario {nombre_de_hoja: lista de diccionarios}.
    Cada fila del Excel es un diccionario; la celda vacia queda como None.
"""
try:
  excel_hojas = pd.read_excel(URL_EXCEL, sheet_name=None)
except OSError as error:
  raise ConnectionError(f"No se pudo leer el Excel: {error}") from error


for nombre_hoja in EXCEL_HOJAS_PRINCIPALES:
  if nombre_hoja not in excel_hojas:
      raise ValueError(f"El Excel no tiene la hoja {nombre_hoja}")

  df = excel_hojas[nombre_hoja]

  # Reemplaza donde pd.isna() sea True *****one
  negocio_excel[nombre_hoja] = df.mask(pd.isna(df), None)

# Mostrar las hojas
print("HOJAS PRINCIPALES DEL EXCEL")
print("=" * 60)
for nombre_hoja in negocio_excel:
    print("*", nombre_hoja)

print("\n\nMOSTRANDO EMPLEADOS")
print("=" * 60)
display(negocio_excel["Empleados"])


HOJAS PRINCIPALES DEL EXCEL
* Empleados
* Puestos
* Vacaciones
* Programacion_Vacaciones
* Ofertas_Compensacion
* Parametros_Vacaciones
* Parametros_Criticidad
* Niveles_Criticidad


MOSTRANDO EMPLEADOS


,id,nombre,puesto_id,salario,fecha_ingreso
0,1,Verónica Ramírez,P18,3450,2026-07-20
1,2,Ricardo Ramírez,P11,4550,2023-08-26
2,3,Karen López,P01,4600,2022-08-07
3,4,Elena Quispe,P19,5850,2025-08-31
4,5,Renzo Vega,P13,14600,2018-01-25
...,...,...,...,...,...
995,996,Milagros Núñez,P16,8050,2021-08-01
996,997,María Lazo,P11,5100,2023-03-14
997,998,Daniela Salazar,P07,18450,2020-04-30
998,999,Ricardo Ramos,P09,11050,2021-01-24


Validaciones

In [4]:
# Iteramos hoja por hoja
for nombre_hoja, df in negocio_excel.items():

    # 1. Duplicados
    cantidad_duplicados = df.duplicated().sum()
    if cantidad_duplicados > 0:
        print(f"Hoja {nombre_hoja}: {cantidad_duplicados} duplicados.")
        df = df.drop_duplicates()

    # 2. Nulos (excluyendo NULOS_PERMITIDOS)
    columnas_obligatorias = [
        col
        for col in df.columns
        if col.lower() not in [n.lower() for n in NULOS_PERMITIDOS]
    ]
    cantidad_nulos = df[columnas_obligatorias].isna().sum().sum()
    if cantidad_nulos > 0:
        print(f"Hoja {nombre_hoja}: {cantidad_nulos} nulos no permitidos.")
        df = df.dropna(subset=columnas_obligatorias)

    # Guardamos la hoja limpia de vuelta en el diccionario
    negocio_excel[nombre_hoja] = df.reset_index(drop=True)


In [5]:
# ==================================================================
# PASO 1: El empleado, la fecha de corte y las reglas de pago
# ==================================================================
# Se busca al empleado por su id y se leen las reglas de pago
# (descuento, sueldos por concepto, etc.) de la hoja Parametros_Vacaciones.

empleado_id = 91             # id del empleado a consultar,
fecha_corte = "2026-10-12"   # un período vence si su fecha de fin es anterior a esta fecha

# Textos que usan las hojas del Excel (se escriben tal cual estan)
TIPO_VENTA = "Venta_50"
RESPUESTA_ACEPTO = "Acepto"
ESTADO_PROGRAMADA = "Programada"
DECISION_APROBADA = "Aprobada"
TIPO_INDEMNIZACION = "Indemnizacion"
OFERTA_PAGADA = "Pagado"
OFERTA_AVISADA = "Avisado"

# Data del empleado
df_empleados = negocio_excel["Empleados"]
df_empleado = df_empleados[df_empleados["id"] == empleado_id]   # filtrado condicional: solo la fila de este empleado

print(f"\n\nMostrando al empleado: {empleado_id}")
print("="*140)
if df_empleado.empty:
    raise ValueError(f"No existe el empleado {empleado_id}")   # detiene la celda: sin empleado no hay boleta
display(df_empleado)
empleado = df_empleado.to_dict(orient="records")[0]   # el empleado como diccionario (para leer su salario)

# Las reglas de pago
df_parametros = negocio_excel["Parametros_Vacaciones"]

parametros = {}   # diccionario {nombre de la regla: valor}
for fila in df_parametros.to_dict(orient="records"):
    parametros[fila["regla"]] = int(fila["valor"])

print("\n\nReglas de pago (hoja Parametros_Vacaciones):")
print("="*140)
display(df_parametros[["regla", "valor", "descripcion"]])



Mostrando al empleado: 91


,id,nombre,puesto_id,salario,fecha_ingreso
90,91,Gabriela Carrasco,P09,10800,2023-09-16




Reglas de pago (hoja Parametros_Vacaciones):


,regla,valor,descripcion
0,dias_vacaciones_periodo,30,Días de vacaciones generados por período
1,meses_generacion_periodo,12,Meses de servicio necesarios para generar el p...
2,meses_periodo_goce,12,Ventana para gozar el período desde que se gen...
3,porcentaje_venta,50,Porcentaje máximo de venta/compensación
4,periodos_criticos,2,Períodos acumulados para priorizar puestos crí...
5,sueldos_indemnizacion,1,Sueldos íntegros SIN descuentos por indemnizac...
6,sueldos_remuneracion_vacacional,1,Sueldos CON descuentos por la remuneración vac...
7,sueldos_mes_goce,1,Sueldos CON descuentos por el sueldo del mes (...
8,porcentaje_descuento,20,Descuento aproximado sobre los sueldos afectos...
9,dias_alerta_vencimiento,90,Días antes de fecha_fin_periodo en que el sist...


In [6]:
# ==================================================================
# PASO 2: Los períodos del empleado y si es indemnizable o no
# ==================================================================
# Días sin cubrir = días pendientes - días vendidos - días programados y aprobados
#   * vendidos:    ofertas Venta_50 que el empleado aceptó
#   * programados: bloques "Programada" con decisión "Aprobada" de RR. HH.
# Un período está Vencido si su fecha de fin es anterior a la fecha de corte.
# Es INDEMNIZABLE si está Vencido y todavía tiene días sin cubrir.

# Períodos del empleado, del más antiguo al más nuevo
df_periodos = negocio_excel["Vacaciones"]
periodos = df_periodos[df_periodos["empleado_id"] == empleado_id].to_dict(orient="records")
periodos.sort(key=lambda periodo: periodo["periodo"])   # lambda: ordena por el año del período

# Sus ofertas y sus bloques de vacaciones programadas
df_ofertas = negocio_excel["Ofertas_Compensacion"]
ofertas = df_ofertas[df_ofertas["empleado_id"] == empleado_id].to_dict(orient="records")

df_programacion = negocio_excel["Programacion_Vacaciones"]
programacion = df_programacion[df_programacion["empleado_id"] == empleado_id].to_dict(orient="records")

filas_periodos = []
for periodo in periodos:
    id_periodo = periodo["vacacion_id"]

    # Días vendidos de este período
    dias_vendidos = 0
    for oferta in ofertas:
        if (oferta["vacacion_id"] == id_periodo
                and oferta["tipo_oferta"] == TIPO_VENTA
                and oferta["estado"] == RESPUESTA_ACEPTO):
            dias_vendidos += oferta["dias"]

    # Días programados y aprobados de este período
    dias_programados = 0
    for bloque in programacion:
        if (bloque["vacacion_id"] == id_periodo
                and bloque["estado"] == ESTADO_PROGRAMADA
                and bloque["decision_rrhh"] == DECISION_APROBADA):
            dias_programados += bloque["dias"]

    # Lo que falta cubrir (nunca baja de 0)
    dias_sin_cubrir = max(0, periodo["dias_pendientes"] - dias_vendidos - dias_programados)

    # La fecha se pasa a texto AAAA-MM-DD, que se puede comparar directamente
    # str(...)[:10] deja la fecha como texto AAAA-MM-DD (quita la hora si la trae)
    # y así se puede comparar con fecha_corte. Ej: "2025-09-15 00:00:00" -> "2025-09-15"
    fecha_fin = str(periodo["fecha_fin_periodo"])[:10]
    if fecha_fin < fecha_corte:
        estado = "Vencido"
    else:
        estado = "Vigente"

    filas_periodos.append({
        "periodo": periodo["periodo"],
        "vacacion_id": id_periodo,
        "fecha_fin": fecha_fin,
        "dias_pendientes": periodo["dias_pendientes"],
        "dias_vendidos": dias_vendidos,
        "dias_programados": dias_programados,
        "dias_sin_cubrir": dias_sin_cubrir,
        "estado": estado,
        "indemnizable": estado == "Vencido" and dias_sin_cubrir > 0,
    })

print(f"\n\nPeríodos del empleado {empleado_id} (fecha de corte {fecha_corte}):")
print("="*140)
display(pd.DataFrame(filas_periodos))




Períodos del empleado 91 (fecha de corte 2026-10-12):


,periodo,vacacion_id,fecha_fin,dias_pendientes,dias_vendidos,dias_programados,dias_sin_cubrir,estado,indemnizable
0,2024,322,2025-09-15,15,0,0,15,Vencido,True
1,2025,323,2026-09-15,30,0,0,30,Vencido,True
2,2026,324,2027-09-15,30,0,0,30,Vigente,False


In [7]:
# ==================================================================
# PASO 3: HISTORIAL: los períodos indemnizables y cómo va su pago
# ==================================================================
# El historial muestra TODO lo que el empleado tiene, incluso lo ya pagado.
# La gestión de cada período indemnizable puede ser:
#   * "Pagada":       ya se le pagó -> NO entra en la boleta
#   * "Avisada":      RR. HH. ya le avisó, falta pagar
#   * "Falta avisar": todavía no se le avisa

historial = []
for fila in filas_periodos:
    if not fila["indemnizable"]:
        continue   # solo interesan los períodos vencidos con días sin cubrir

    gestion = "Falta avisar"
    for oferta in ofertas:
        if oferta["vacacion_id"] == fila["vacacion_id"] and oferta["tipo_oferta"] == TIPO_INDEMNIZACION:
            if oferta["estado"] == OFERTA_PAGADA:
                gestion = "Pagada"
            elif oferta["estado"] == OFERTA_AVISADA and gestion != "Pagada":
                gestion = "Avisada"   # Pagada tiene prioridad sobre Avisada

    historial.append({
        "periodo": fila["periodo"],
        "vacacion_id": fila["vacacion_id"],
        "dias_sin_cubrir": fila["dias_sin_cubrir"],
        "gestion": gestion,
    })

print(f"\n\nHistorial de indemnización del empleado {empleado_id}:")
print("="*140)
if historial:
    display(pd.DataFrame(historial))
else:
    print("Sin períodos indemnizables")




Historial de indemnización del empleado 91:


,periodo,vacacion_id,dias_sin_cubrir,gestion
0,2024,322,15,Falta avisar
1,2025,323,30,Avisada


In [8]:
# ==================================================================
# PASO 4: La BOLETA del próximo pago (lo que se le paga ahora)
# ==================================================================
# Orden de la boleta:
#   1. Sueldo del mes: el mes COMPLETO, UNA sola vez, con descuento.
#   2. Por cada período pendiente (del más antiguo al más nuevo):
#        - Remuneración vacacional = sueldo x factor, CON descuento
#        - Indemnización           = sueldo x factor, SIN descuento
#      donde factor = días sin cubrir / 30 (la parte del período que falta)
# Los períodos "Pagada" no entran: ya se pagaron.

salario = empleado["salario"]
descuento = parametros["porcentaje_descuento"]   # 20 (%)

lineas_boleta = []

# 1. Sueldo del mes (una sola vez, sin factor)
sueldo_bruto = round(salario * parametros["sueldos_mes_goce"], 2)
sueldo_descuento = round(sueldo_bruto * descuento / 100, 2)
lineas_boleta.append({
    "concepto": "Sueldo del mes",
    "periodo": "-",
    "dias": "-",
    "gestion": "-",
    "bruto": sueldo_bruto,
    "descuento": sueldo_descuento,
    "neto": round(sueldo_bruto - sueldo_descuento, 2),
})

# 2. Remuneración e indemnización de cada período pendiente
for periodo in historial:
    if periodo["gestion"] == "Pagada":
        continue   # ya se pagó: no forma parte de la boleta

    factor = periodo["dias_sin_cubrir"] / parametros["dias_vacaciones_periodo"]

    remuneracion = round(salario * parametros["sueldos_remuneracion_vacacional"] * factor, 2)
    remuneracion_descuento = round(remuneracion * descuento / 100, 2)
    indemnizacion = round(salario * parametros["sueldos_indemnizacion"] * factor, 2)

    lineas_boleta.append({
        "concepto": "Remuneración vacacional",
        "periodo": periodo["periodo"],
        "dias": periodo["dias_sin_cubrir"],
        "gestion": periodo["gestion"],
        "bruto": remuneracion,
        "descuento": remuneracion_descuento,
        "neto": round(remuneracion - remuneracion_descuento, 2),
    })
    lineas_boleta.append({
        "concepto": "Indemnización vacacional",
        "periodo": periodo["periodo"],
        "dias": periodo["dias_sin_cubrir"],
        "gestion": periodo["gestion"],
        "bruto": indemnizacion,
        "descuento": 0.0,   # la indemnización no lleva descuento
        "neto": indemnizacion,
    })

print(f"\n\nBoleta del próximo pago del empleado {empleado_id}:")
print("="*140)
display(pd.DataFrame(lineas_boleta))




Boleta del próximo pago del empleado 91:


,concepto,periodo,dias,gestion,bruto,descuento,neto
0,Sueldo del mes,-,-,-,10800.0,2160.0,8640.0
1,Remuneración vacacional,2024,15,Falta avisar,5400.0,1080.0,4320.0
2,Indemnización vacacional,2024,15,Falta avisar,5400.0,0.0,5400.0
3,Remuneración vacacional,2025,30,Avisada,10800.0,2160.0,8640.0
4,Indemnización vacacional,2025,30,Avisada,10800.0,0.0,10800.0


In [9]:
# ==================================================================
# PASO 5: Los totales de la boleta
# ==================================================================
# Un acumulador es una variable que empieza en 0 y va sumando dentro de un bucle.

total_bruto = 0
total_descuento = 0
total_neto = 0
for linea in lineas_boleta:
    total_bruto += linea["bruto"]
    total_descuento += linea["descuento"]
    total_neto += linea["neto"]

# Días de indemnización que se pagan en esta boleta (sin contar los ya pagados)
dias_a_pagar = 0
for periodo in historial:
    if periodo["gestion"] != "Pagada":
        dias_a_pagar += periodo["dias_sin_cubrir"]

print(f"\n\nTotales de la boleta del empleado {empleado_id}:")
print("="*140)
print(f"TOTAL -> bruto: S/ {total_bruto:,.2f} | descuento: S/ {total_descuento:,.2f} | neto: S/ {total_neto:,.2f}")
print(f"Días de indemnización a pagar: {dias_a_pagar}")




Totales de la boleta del empleado 91:
TOTAL -> bruto: S/ 43,200.00 | descuento: S/ 5,400.00 | neto: S/ 37,800.00
Días de indemnización a pagar: 45
